[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/ai-ml-interviews-study-guide/blob/main/notebooks/week3_power_analysis.ipynb)


# Week 3 — Power & Sample Size for Proportions

**Reference:** Wasserman, *All of Statistics*, Ch. 10.6 (Power).

## Why this matters

**Statistical power** is the probability of rejecting the null hypothesis when the alternative is true:

$$\text{power} = 1 - \beta = P(\text{reject } H_0 \mid H_1 \text{ true})$$

An eval that is too small can miss real differences (low power) and an eval that is too big wastes compute. Power analysis tells you the minimum sample size required to detect an effect of a given size.

## Interview payoff

When an interviewer asks *"How would you design an evaluation for X?"*, the strongest answers anchor on a sample-size calculation. You want to be able to say:

> "To detect a 5 percentage point difference in attack success rate at 80% power and a two-sided alpha of 0.05, I need roughly N tasks per arm. Here is how I'd derive that."

## Scenario

**SandboxBench** is a synthetic agentic-safety benchmark. We are comparing two agents on the same set of tasks and want to know whether Agent B has a meaningfully lower attack success rate (ASR) than Agent A.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(13)

## Scenario parameters

- Agent A baseline ASR: $p_1 = 0.30$
- Agent B target ASR: $p_2 = 0.25$ (a 5 percentage point reduction)
- Two-sided $\alpha = 0.05$
- Target power $1 - \beta = 0.80$

We treat the two agents as independent samples (each task is run independently per agent). A paired analysis would be more powerful when the same tasks are run on both, but the unpaired version is the standard back-of-the-envelope.

In [ ]:
p1 = 0.30          # Agent A ASR (baseline)
p2 = 0.25          # Agent B ASR (target)
alpha = 0.05       # two-sided significance
power_target = 0.80

delta = p1 - p2
print(f"p1 = {p1}, p2 = {p2}, effect (delta) = {delta:.3f}")
print(f"alpha = {alpha} (two-sided), target power = {power_target}")

## Analytic sample size (two-proportion z-test)

For a two-sided test of $H_0 : p_1 = p_2$ versus $H_1 : p_1 \ne p_2$, the standard normal-approximation sample size per group is:

$$n = \frac{\left(z_{1-\alpha/2}\sqrt{2\bar{p}(1-\bar{p})} + z_{1-\beta}\sqrt{p_1(1-p_1) + p_2(1-p_2)}\right)^2}{(p_1 - p_2)^2}$$

where $\bar{p} = (p_1 + p_2) / 2$.

Two intuitions:
- The numerator's first term comes from the **null** distribution (pooled variance — the test statistic uses pooling).
- The second term comes from the **alternative** distribution (unpooled variance at the true $p_1, p_2$).
- The denominator is the squared effect size in raw probability units.

In [ ]:
def required_n_per_group(p1, p2, alpha=0.05, power=0.80):
    p_bar = (p1 + p2) / 2
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    num = (z_alpha * np.sqrt(2 * p_bar * (1 - p_bar))
           + z_beta * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    den = (p1 - p2) ** 2
    return num / den

n_exact = required_n_per_group(p1, p2, alpha, power_target)
n_required = int(np.ceil(n_exact))
print(f"Required n per group (exact): {n_exact:.2f}")
print(f"Required n per group (rounded up): {n_required}")
print(f"Total tasks across both agents: {2 * n_required}")

## Simulation verification

We will Monte Carlo the experiment 5,000 times at $n = n_{\text{required}}$ per group and check the empirical rejection rate. It should be close to 0.80. Small discrepancies are expected because:

1. The formula is a normal approximation (we are not applying any continuity correction).
2. We rounded $n$ up.
3. Monte Carlo noise at 5,000 reps is roughly $\pm 0.011$ on a power around 0.80.

In [ ]:
def two_prop_z_test(x1, n1, x2, n2):
    """Pooled two-proportion z statistic. Returns z; caller compares |z| to z_{1-alpha/2}."""
    p_hat1 = x1 / n1
    p_hat2 = x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    # Avoid division by zero in degenerate Monte Carlo draws.
    se = np.where(se == 0, np.nan, se)
    return (p_hat1 - p_hat2) / se

def simulate_power(n, p1, p2, alpha=0.05, reps=5000, rng=None):
    rng = np.random.default_rng(rng)
    x1 = rng.binomial(n, p1, size=reps)
    x2 = rng.binomial(n, p2, size=reps)
    z = two_prop_z_test(x1, n, x2, n)
    crit = norm.ppf(1 - alpha / 2)
    rejected = np.abs(z) > crit
    return np.nanmean(rejected)

emp_power = simulate_power(n_required, p1, p2, alpha=alpha, reps=5000, rng=13)
print(f"Empirical power at n={n_required}: {emp_power:.3f}")
print(f"Target power: {power_target:.3f}")
print(f"Discrepancy: {emp_power - power_target:+.3f}")

## Power curve vs sample size

Sweep $n$ over a grid and compare the **empirical** power (Monte Carlo, 1,000 reps each) against the **analytic** power from the same normal approximation.

Analytic power for a two-sided two-proportion test:

$$1 - \beta \approx \Phi\left(\frac{|p_1 - p_2| \sqrt{n} - z_{1-\alpha/2}\sqrt{2\bar{p}(1-\bar{p})}}{\sqrt{p_1(1-p_1) + p_2(1-p_2)}}\right)$$

In [ ]:
def analytic_power(n, p1, p2, alpha=0.05):
    p_bar = (p1 + p2) / 2
    z_alpha = norm.ppf(1 - alpha / 2)
    num = abs(p1 - p2) * np.sqrt(n) - z_alpha * np.sqrt(2 * p_bar * (1 - p_bar))
    den = np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))
    return norm.cdf(num / den)

ns = np.array([50, 100, 200, 400, 800, 1600, 3200])
rng = np.random.default_rng(13)
emp = np.array([simulate_power(n, p1, p2, alpha=alpha, reps=1000, rng=rng) for n in ns])
ana = np.array([analytic_power(n, p1, p2, alpha=alpha) for n in ns])

for n, e, a in zip(ns, emp, ana):
    print(f"n={n:5d}  empirical={e:.3f}  analytic={a:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ns, ana, 'o-', label='Analytic power', linewidth=2)
ax.plot(ns, emp, 's--', label='Empirical power (1000 reps)', linewidth=2)
ax.axhline(power_target, color='red', linestyle=':', label=f'Target power = {power_target}')
ax.axvline(n_required, color='green', linestyle=':', label=f'Analytic n = {n_required}')
ax.set_xscale('log')
ax.set_xlabel('n per group')
ax.set_ylabel('Power')
ax.set_title(f'Power vs sample size — detecting p1={p1}, p2={p2}')
ax.set_ylim(0, 1.02)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Effect-size sensitivity

Now we fix $n$ at the analytic answer above and sweep the **true** ASR gap. The headline lesson: small effects fall off a cliff. Halving the effect size roughly **quadruples** the required sample, so a fixed-$n$ eval has very low power against subtle differences.

In [ ]:
gaps = np.array([0.01, 0.02, 0.03, 0.05, 0.08, 0.10])
rng = np.random.default_rng(13)
powers = []
for g in gaps:
    p2_g = p1 - g
    powers.append(simulate_power(n_required, p1, p2_g, alpha=alpha, reps=2000, rng=rng))
powers = np.array(powers)

for g, pw in zip(gaps, powers):
    print(f"true gap={g:.2f}  empirical power={pw:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(gaps, powers, 'o-', linewidth=2)
ax.axhline(power_target, color='red', linestyle=':', label=f'Target power = {power_target}')
ax.set_xlabel('True ASR gap (p1 - p2)')
ax.set_ylabel('Empirical power')
ax.set_title(f'Effect-size sensitivity at n={n_required} per group')
ax.set_ylim(0, 1.02)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Decision: "Do I have enough samples?"

Suppose SandboxBench actually ships with **N = 150 tasks per agent**. The right framing is not *"is this enough?"* but *"what is the smallest effect this eval can detect with 80% power?"* — the **minimum detectable effect (MDE)**.

We solve for the gap $\delta$ that yields power $= 0.80$ at $n = 150$ by binary search on the analytic power function (anchored at $p_1 = 0.30$, with $p_2 = p_1 - \delta$).

In [ ]:
def mde(n, p1, alpha=0.05, power=0.80, lo=1e-4, hi=None, tol=1e-5):
    if hi is None:
        hi = p1  # cannot reduce ASR below 0
    for _ in range(200):
        mid = (lo + hi) / 2
        pw = analytic_power(n, p1, p1 - mid, alpha=alpha)
        if pw < power:
            lo = mid
        else:
            hi = mid
        if hi - lo < tol:
            break
    return (lo + hi) / 2

n_ship = 150
mde_150 = mde(n_ship, p1, alpha=alpha, power=power_target)
print(f"With n={n_ship} per agent, the minimum detectable ASR gap at 80% power is {mde_150:.4f}")
print(f"  -> roughly {mde_150 * 100:.1f} percentage points")

# Sanity-check via simulation.
emp = simulate_power(n_ship, p1, p1 - mde_150, alpha=alpha, reps=5000, rng=13)
print(f"  Empirical power at that gap: {emp:.3f} (should be ~0.80)")

print()
print("=" * 60)
print(f"VERDICT: With n={n_ship}, you can detect an ASR difference of "
      f"{mde_150 * 100:.1f}% or larger with 80% power.")
print(f"The original 5% target is {'detectable' if mde_150 <= 0.05 else 'NOT detectable'} at this sample size.")
print("=" * 60)

## Flashcard summary

**Question.** SandboxBench has $N$ tasks per agent. I want to detect a 5 percentage-point ASR difference between two agents (Agent A baseline ASR = 30%). Two-sided $\alpha = 0.05$, power = 0.80. How many tasks do I need per agent?

**Formula.**

$$n = \frac{\left(z_{1-\alpha/2}\sqrt{2\bar{p}(1-\bar{p})} + z_{1-\beta}\sqrt{p_1(1-p_1) + p_2(1-p_2)}\right)^2}{(p_1 - p_2)^2}$$

**Plug in.** $p_1 = 0.30$, $p_2 = 0.25$, $\bar{p} = 0.275$, $z_{0.975} \approx 1.96$, $z_{0.80} \approx 0.84$.

**Answer.** Roughly **n ≈ 1,250 per agent** (≈ 2,500 tasks total). At a shipped size of $n = 150$, the minimum detectable effect is around **14 percentage points** — nowhere near 5%.

## Interview rehearsal

**Q:** *"How would you design an evaluation to compare the safety of two agents on a benchmark like SandboxBench?"*

**90-second model answer:**

> The first thing I would pin down is the smallest effect we actually care about. For an attack-success-rate metric, suppose the baseline agent sits at 30% and we want to detect a 5-percentage-point reduction with 80% power and a two-sided alpha of 0.05. Using the standard normal-approximation formula for a two-proportion z-test, that works out to roughly 1,250 tasks per agent — so about 2,500 evaluations total.
>
> If our benchmark only has 150 tasks per agent, we are not in a position to detect a 5-point difference; the minimum detectable effect at that sample size is more like 14 percentage points. So I'd do one of three things: (1) expand the benchmark, (2) use paired evaluation on the same tasks to reduce variance, or (3) be honest in the writeup that we are powered only for large effects, and report a confidence interval rather than a p-value.
>
> The general principle: every eval design question should start with a power calculation. It forces you to commit to an effect size and prevents the common mistake of running an underpowered study and over-interpreting a non-significant result as evidence of no effect."